In [6]:
# Создадим тестовый файл 'test_data.csv'
csv_content = """name,gdp
Russian Federation,11000.50
Luxembourg,90000.00
United States,60000.00
Country With No Data,
Burundi,300.00
Monaco,180000.00
"""

with open('test_data.csv', 'w', encoding='utf-8') as f:
    f.write(csv_content)

print("Файл test_data.csv создан. Теперь запускайте основной код!")

Файл test_data.csv создан. Теперь запускайте основной код!


In [10]:
# Задание task_03_04_07.
# Выполнил: Гришина А.А.
# Группа: ЦИБ-251

import csv

class NoSuchCountryError(Exception):
    def __init__(self, message):
        super().__init__(message)

class IllegalArgumentError(ValueError):
    pass

def load_data(filename):
    """Загрузить данные ВВП на душу населения из csv-файла 'filename'.
    Если значения для какого-либо государства не известно, строка должна
    быть пропущена и отсутствовать в результате.
    """
    result = []
    with open(filename, 'r', encoding='utf-8') as f:
        # Предполагаем, что в файле есть заголовки name и gdp
        reader = csv.DictReader(f)
        for row in reader:
            try:
                # Проверяем, есть ли значение gdp и не пустое ли оно
                gdp_val = row.get('gdp')
                if gdp_val is not None and gdp_val.strip() != '':
                    result.append({
                        'name': row['name'],
                        'gdp': float(gdp_val)
                    })
            except (ValueError, TypeError):
                # Если значение не конвертируется в float, пропускаем строку
                pass
    return result

def search(data, criteria):
    """Выполнить поиск государства-значения в 'data' по критерию 'criteria'."""

    # Определение операции
    if criteria == "-max-":
        if not data:
            raise NoSuchCountryError("Список данных пуст, невозможно найти максимум.")
        # Поиск страны с максимальным ВВП
        return max(data, key=lambda x: x['gdp'])

    elif criteria == "-min-":
        if not data:
            raise NoSuchCountryError("Список данных пуст, невозможно найти минимум.")
        # Поиск страны с минимальным ВВП
        return min(data, key=lambda x: x['gdp'])

    else:
        # Поиск по названию страны
        for country in data:
            if country['name'] == criteria:
                return country
        raise NoSuchCountryError(f"Страна '{criteria}' не найдена в данных.")

def save_data(filename, data, criteria):
    """Сохранить данные 'data' в csv-файл 'filename' по критерию 'criteria'."""
    try:
        # Разбираем критерий (например, "top=5" -> method="top", value="5")
        if "=" not in criteria:
            raise ValueError("Критерий должен содержать знак '='")

        method, value_str = criteria.split("=")
        method = method.strip()
        value_str = value_str.strip()

        filtered_data = []

        # Определение операции
        if method == "top":
            count = int(value_str)
            if count <= 0: raise ValueError("Число должно быть больше 0")
            # Сортировка по убыванию (reverse=True) и взятие первых count
            sorted_data = sorted(data, key=lambda x: x['gdp'], reverse=True)
            filtered_data = sorted_data[:count]

        elif method == "tail":
            count = int(value_str)
            if count <= 0: raise ValueError("Число должно быть больше 0")
            # Сортировка по возрастанию и взятие первых count
            sorted_data = sorted(data, key=lambda x: x['gdp'])
            filtered_data = sorted_data[:count]

        elif method == "greater":
            threshold = float(value_str)
            # Отбор тех, кто больше threshold, сортировка по убыванию
            filtered_data = [x for x in data if x['gdp'] > threshold]
            filtered_data = sorted(filtered_data, key=lambda x: x['gdp'], reverse=True)

        elif method == "less":
            threshold = float(value_str)
            # Отбор тех, кто меньше threshold, сортировка по возрастанию
            filtered_data = [x for x in data if x['gdp'] < threshold]
            filtered_data = sorted(filtered_data, key=lambda x: x['gdp'])

        else:
            raise ValueError(f"Неизвестный метод: {method}")

        # Сохранение в файл
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            fieldnames = ['name', 'gdp']
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(filtered_data)

    except Exception as err:
        raise IllegalArgumentError(
             "Значение параметра 'criteria' содержит недопустимое значение или формат. "
             "Используйте: top=X, tail=X, greater=X, less=X")

# ==========================================
# Основная программа с обработкой исключений
# ==========================================
try:
    # ВАЖНО: Убрали жесткое присваивание, теперь программа ждет ввода
    filename = input("Введите имя файла для загрузки: ")
    save_filename = input("Введите имя файла для сохранения (например, output.csv): ")

    # 1. Загрузка
    data = load_data(filename)
    print(f"\nУспешно загружено записей: {len(data)}")

    # 2. Поиск
    print("\nСтрана с максимальным ВВП:", search(data, criteria="-max-"))
    print("Страна с минимальным ВВП:", search(data, criteria="-min-"))

    # Попытка найти Россию (может не быть в файле, тогда будет ошибка)
    try:
        print("Поиск Russia:", search(data, criteria="Russian Federation"))
    except NoSuchCountryError:
        print("Поиск Russia: Страна не найдена (возможно, в файле другое название).")

    # 3. Сохранение разных выборок
    # Сохраняем топ-5 в указанный файл
    save_data(save_filename, data, criteria="top=5")
    print(f"\nТоп-5 стран сохранен в файл: {save_filename}")

    # Для демонстрации сохраняем остальные варианты в разные файлы
    save_data("tail_5.csv", data, criteria="tail=5")
    print("Хвост (5 стран с мин. ВВП) сохранен в tail_5.csv")

    save_data("greater_5000.csv", data, criteria="greater=5000.50")
    print("Страны > 5000.50 сохранены в greater_5000.csv")

    save_data("less_5000.csv", data, criteria="less=5000.50")
    print("Страны < 5000.50 сохранены в less_5000.csv")

except NoSuchCountryError as e:
    print(f"\nОшибка поиска: {e}")
except IllegalArgumentError as e:
    print(f"\nОшибка аргумента: {e}")
except FileNotFoundError as e:
    print(f"\nОшибка файла: {e} (Проверьте имя файла и убедитесь, что он загружен в Colab)")
except Exception as e:
    print(f"\nПроизошла непредвиденная ошибка: {e}")



Введите имя файла для загрузки: test_data.csv
Введите имя файла для сохранения (например, output.csv): output.csv

Успешно загружено записей: 5

Страна с максимальным ВВП: {'name': 'Monaco', 'gdp': 180000.0}
Страна с минимальным ВВП: {'name': 'Burundi', 'gdp': 300.0}
Поиск Russia: {'name': 'Russian Federation', 'gdp': 11000.5}

Топ-5 стран сохранен в файл: output.csv
Хвост (5 стран с мин. ВВП) сохранен в tail_5.csv
Страны > 5000.50 сохранены в greater_5000.csv
Страны < 5000.50 сохранены в less_5000.csv
